# 02 - Feature Analysis

Compare MFCC and Mel-Spectrogram representations. Visualize SpecAugment effects.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
from src.config import FeatureConfig, AugmentConfig
from src.data.features import FeatureExtractor
from src.data.augmentation import SpecAugmentor
from src.data.download import prepare_dataset

plt.style.use('seaborn-v0_8-paper')

In [ ]:
# Load a sample audio
metadata = prepare_dataset('../data', num_speakers=10, min_utterances=5)
sample_path = metadata.iloc[0]['file_path']
waveform, sr = torchaudio.load(sample_path, backend="soundfile")
print(f"Waveform shape: {waveform.shape}, Sample rate: {sr}")

In [ ]:
# Extract features
mel_config = FeatureConfig(feature_type='mel_spectrogram')
mfcc_config = FeatureConfig(feature_type='mfcc')

mel_extractor = FeatureExtractor(mel_config, sample_rate=16000)
mfcc_extractor = FeatureExtractor(mfcc_config, sample_rate=16000)

mel_spec = mel_extractor(waveform)
mfcc = mfcc_extractor(waveform)

print(f"Mel-spectrogram shape: {mel_spec.shape}")
print(f"MFCC shape: {mfcc.shape}")

In [ ]:
# Side-by-side comparison
from src.evaluation.visualization import plot_spectrogram_examples

plot_spectrogram_examples(
    waveform=waveform[0].numpy(),
    mel_spectrogram=mel_spec[0].numpy(),
    mfcc=mfcc.numpy(),
    save_path='../results/feature_comparison.png',
)

In [ ]:
# SpecAugment demonstration
aug_config = AugmentConfig(spec_augment=True, freq_mask_param=15, time_mask_param=20, num_masks=2)
spec_aug = SpecAugmentor(aug_config)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(mel_spec[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Original Mel-Spectrogram', fontsize=14)

for i in range(1, 3):
    augmented = spec_aug(mel_spec.clone())
    axes[i].imshow(augmented[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
    axes[i].set_title(f'SpecAugment #{i}', fontsize=14)

for ax in axes:
    ax.set_xlabel('Time Frame')
    ax.set_ylabel('Mel Band')

plt.tight_layout()
plt.savefig('../results/specaugment_demo.png', dpi=300)
plt.show()